# Workshop 4 (CBA)

In [ ]:
import os
import shutil
import zipfile
from datetime import datetime
from pathlib import Path
from urllib.request import urlretrieve

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import pandas as pd
import pypsa
from IPython.display import SVG, Code, IFrame, Image, display
from pdf2image import convert_from_path
from pdf2image.exceptions import PDFPageCountError
from pypsa.plot.maps.static import add_legend_lines
from pypsa_explorer import create_app

pypsa.set_option("params.statistics.nice_names", True)
pypsa.set_option("params.statistics.drop_zero", True)
pypsa.set_option("params.statistics.round", 3)
plt.rcParams["figure.figsize"] = [14, 7]
clip = 1  # TWh

## Reminder: The `Snakemake` tool

<img src="https://raw.githubusercontent.com/open-energy-transition/open-tyndp-workshops/792b8474ab5096e5ab8db2822af4fcd9fe659eb6/open-tyndp-workshops/snakemake_logo.png" width="500px" />

The `Snakemake` workflow management system is a tool to create reproducible and scalable data analyses.
Workflows are described via a human readable, Python based language. They can be seamlessly scaled to server, cluster, grid, and cloud environments, without the need to modify the workflow definition.

Snakemake follows the [GNU Make](https://www.gnu.org/software/make) paradigm: workflows are defined in terms of so-called `rules` that specify how to create a set of output files from a set of input files. Dependencies between the rules are determined automatically, creating a DAG (directed acyclic graph) of jobs that can be automatically parallelized.

:::{note}
Documentation for this package is available at https://snakemake.readthedocs.io/. You can also check out a [slide deck Snakemake Tutorial](https://slides.com/johanneskoester/snakemake-tutorial) by Johannes Köster (2024).

Mölder, F., Jablonski, K.P., Letcher, B., Hall, M.B., Tomkins-Tinch, C.H., Sochat, V., Forster, J., Lee, S., Twardziok, S.O., Kanitz, A., Wilm, A., Holtgrewe, M., Rahmann, S., Nahnsen, S., Köster, J., 2021. Sustainable data analysis with Snakemake. F1000Res 10, 33.
:::


### A minimal Snakemake example

To check out how this looks in practice, we've prepared a minimal Snakemake example workflow that processes some data. The minimal workflow consists of the following rules:
- `retrieve_data`
- `build_data`
- `prepare_network`
- `solve_network`
- `plot_benchmark`
- `all`

These rules are illustrative and mimic the Open-TYNDP structure and nomenclature.

<center>

![](minimal_workflow.png)

</center>

We have already loaded the raw data file used in this minimal example into our working directory.

As you can see, the `plot_benchmark` rule will be called twice with two different filename extensions. For this, we are taking advantage of the concept of wildcards (`ext`). Snakemake will automatically resolve the wildcards using the dependency graph. In this case, the `all` rule takes as input both a `png` and a `pdf` figure which propagates back throughout the workflow.

#### The `Snakefile` and `rules`

The rules need to be defined in a so-called `Snakefile` that sits in your current working directory. For our minimal example the `Snakefile` looks like this:

In [ ]:
Code(filename="Snakefile", language="Python")

You can check out the scripts under `scripts`. You will see that they are simplistic and only serve an illustrative purpose.

You can also observe how the `plot_benchmark` rule is defined to take advantage of the wildcards. This reduces the redundancy in the `Snakefile`. Wildcards are defined between `{ }` in the rule definition.

#### Calling a workflow

You can trigger the workflow by specifying a target file, like `data/benchmark.pdf`, or any intermediate file:
```bash
snakemake -call data/benchmark.pdf
```

Alternatively, you can also execute the workflow by calling a rule that produces an intermediate file:
```bash
snakemake -call build_data
```
**NOTE:** You cannot call a rule that includes a wildcard without specifying what the wildcard should be filled with. Otherwise, Snakemake will not know what to propagate back.

Or you can call the common rule `all` which can be used to execute the entire workflow. It takes the final workflow output as its input and thus requires all previous dependent rules to be run as well:
```bash
snakemake -call all
```

Because we defined the `all` rule as first in the `Snakefile`, this rule is assumed to be the default and the following also works:
```bash
snakemake -call
```

A very important feature is the `-n` flag which executes a `dry-run`. It is recommended to always first execute a `dry-run` before the actual execution of a workflow. This simply prints out the DAG of the workflow to investigate without actually executing it.

Let's try this out and investigate the output:

In [ ]:
! snakemake -call -n

As you can see, the `plot_benchmark` rule will be executed twice due to wildcards.

#### Visualizing the `DAG` of a workflow

You can also visualize the DAG of jobs using the `--dag` flag and the Graphviz `dot` command. This will not run the workflow but only create the visualization:

In [ ]:
! snakemake -call --dag | python scripts/filter_dag.py | dot -Tpng -o dag_minimal.png
# For Windows run instead:
# ! snakemake -call --dag | python scripts\\filter_dag.py | dot -Tpng -o dag_minimal.png

In [ ]:
Image("dag_minimal.png")

Rules that need to be executed will be presented as plain lines, while those that have already been executed will be presented as dotted lines. An alternative to the DAG is the `rulegraph`. This graph is typically less crowded as you will only visualize the dependency graph of rules. This representation is leaner than the DAG because rules are not repeated for wildcards.

In [ ]:
! snakemake -call all --rulegraph | python scripts/filter_dag.py | dot -Tpng -o rulegraph_minimal.png
# For Windows run instead:
# ! snakemake -call all --rulegraph | python scripts\\filter_dag.py | dot -Tpng -o rulegraph_minimal.png

In [ ]:
Image("rulegraph_minimal.png")

As you can see, the `plot_benchmark` rule is only represented once.

Alternatively, you can also visualize a `filegraph`, which is similar to the `rulegraph` but includes some information about the inputs and outputs to each of the rules.

In [ ]:
! snakemake -call all --filegraph | python scripts/filter_dag.py | dot -Tsvg -o filegraph_minimal.svg
# For Windows run instead:
# ! snakemake -call all --filegraph | python scripts\\filter_dag.py | dot -Tsvg -o filegraph_minimal.svg

In [ ]:
SVG("filegraph_minimal.svg")

### Task 1: Executing a workflow with Snakemake

**a)** For our minimal example, execute a `dry-run` to produce the intermediate file `data/base_2030.nc`.

**b)** Execute the entire workflow and investigate what happens if you try to execute the workflow again.

**c)** Delete the final output file `data/benchmark.pdf` and investigate what happens if you try to execute the workflow again.

**d)** Change a value in the raw input data file `data/data_raw.csv` and save it again, overwriting the original file. Investigate what happens if you try to execute the workflow again.

Hint: You can also just `touch` the file by executing `Path("data/data_raw.csv").touch()`. This will mimic a file edit.

**e)** (Optional) Open the `Snakefile` and add a second rule that processes the file `data_raw_2.csv` using the same script as the `build_data` rule. Add the output of this new rule as a second input to the `prepare_network` rule.

In [ ]:
# Your solution a)

In [ ]:
# Your solution b)

In [ ]:
# Your solution c)

In [ ]:
# Your solution d)

In [ ]:
# Your solution e)

## Exploring Open-TYNDP CBA workflow

### Cloning the Open-TYNDP project

First, navigate into the `data/` folder:

In [ ]:
os.chdir("data/")
print("Directory changed to:", os.getcwd())

Directory changed to: /Users/meas/oet/open-tyndp-workshops/open-tyndp-workshops/data


In [ ]:
# os.chdir("/Users/meas/oet/open-tyndp-workshops/open-tyndp-workshops/data/")

Clone the Open-TYNDP repository directly:

In [ ]:
# TODO: add conditional feature to first check that we're in /data/ and only clone if not already in the correct directory
! git clone https://github.com/open-energy-transition/open-tyndp.git

Cloning into 'open-tyndp'...
remote: Enumerating objects: 43002, done.
remote: Counting objects: 100% (604/604), done.
remote: Compressing objects: 100% (249/249), done.
remote: Total 43002 (delta 514), reused 388 (delta 355), pack-reused 42398 (from 3)
Receiving objects: 100% (43002/43002), 147.35 MiB | 34.45 MiB/s, done.
Resolving deltas: 100% (33499/33499), done.


Now, navigate into the the `open-tyndp` directory:

In [ ]:
os.chdir("open-tyndp/")
print("Directory changed to:", os.getcwd())

Directory changed to: /Users/meas/oet/open-tyndp-workshops/open-tyndp-workshops/data/open-tyndp


In [ ]:
! git checkout cba-low-res

branch 'cba-low-res' set up to track 'origin/cba-low-res'.
Switched to a new branch 'cba-low-res'


In [ ]:
! git status

On branch cba-low-res
Your branch is up to date with 'origin/cba-low-res'.

nothing to commit, working tree clean


(add instruction to install pixi - copy from workshop 3)

Use `pixi` to install the `open-tyndp` environment:

In [ ]:
! pixi install -e open-tyndp

zsh:1: command not found: pixi


In [ ]:
os.environ["PATH"] = os.path.expanduser("~/.pixi/bin") + ":" + os.environ["PATH"]

In [ ]:
! pixi install -e open-tyndp

⠁ preparing packages   [━━━━╾───────────────] 60/545 gurobi (+219)         
⠁ preparing packages   [━━━━━━━━━━━━━━━━━━━╾] 484/545 openssl (+51) 
▪ preparing packages   [━━━━━━━━━━━━━━━━━━━━] 545/545  )             
▪ preparing packages   [━━━━━━━━━━━━━━━━━━━━] 545/545  cosystem (+543
▪ preparing packages   [━━━━━━━━━━━━━━━━━━━━] 545/545  cosystem (+543
▪ preparing packages   [━━━━━━━━━━━━━━━━━━━━] 545/545  cosystem (+538
▪ preparing packages   [━━━━━━━━━━━━━━━━━━━━] 545/545  cosystem (+538
▪ preparing packages   [━━━━━━━━━━━━━━━━━━━━] 545/545  cosystem (+535
▪ preparing packages   [━━━━━━━━━━━━━━━━━━━━] 545/545  cosystem (+534
▪ preparing packages   [━━━━━━━━━━━━━━━━━━━━] 545/545  cosystem (+526
▪ preparing packages   [━━━━━━━━━━━━━━━━━━━━] 545/545  cosystem (+525
▪ preparing packages   [━━━━━━━━━━━━━━━━━━━━] 545/545  cosystem (+523
▪ preparing packages   [━━━━━━━━━━━━━━━━━━━━] 545/545  cosystem (+522
▪ preparing packages   [━━━━━━━━━━━━━━━━━━━━] 545/545  cosystem (+521
▪ preparing pac

### Running a coupled SB->CBA workflow

To run the CBA workflow, the `pixi run tyndp-cba` runs the full process: taking a solved SB network -> performing the TOOT/PINT analysis on a project -> calculating the CBA indicators. 

The default Open-TYNDP settings can be found in the `config/config.tyndp.yaml` file. 

TODO: add chunks of config file (refer to previous workshop about manual adjustments)
- add snippets of default values (runs, horizons, snapshots, projects, etc)

mention:
- because of notebook environment, we are focusing on command line changes, but in reality it's easier to modify in config files directly using dedicated IDE

In [ ]:
# use filter to create viz of DAG for the CBA workflow
# can add this pre-created figure in this section

First, let's check what running the full could look like by doing a dry-run of `pixi run tyndp-cba`:

In [ ]:
! pixi run tyndp-cba -n

⠁ activating environment                                                        kemake -call cba --configfile config/config.tyndp.yaml -n                                                        ⠁                                                                               Using workflow specific profile profiles/default for setting default command line arguments.
Config file config/config.default.yaml is extended by additional config specified via the command line.
Config file config/plotting.default.yaml is extended by additional config specified via the command line.
Config file config/benchmarking.default.yaml is extended by additional config specified via the command line.
host: mm-mbp-2025
Building DAG of jobs...
Job stats:
job                                           count
------------------------------------------  -------
cba                                               1
clean_projects                                   12
collect_cba_scenario                             15
r

We can specify the specific scenario (e.g, `NT`), the project (e.g., `t4`), and planning horizon (e.g., 2030) we want to run also directly in the command line. 

In [ ]:
! pixi run tyndp-cba --config 'run={"name":"NT"}' 'cba={"planning_horizons":[2030],"projects":["t4"]}' -n

⠁ activating environment                                                        kemake -call cba --configfile config/config.tyndp.yaml --config run={"name":"NT"} cba={"planning_horizons":[2030],"projects":["t4"]} -n                                                          ⠁                                                                               Using workflow specific profile profiles/default for setting default command line arguments.
Config file config/config.default.yaml is extended by additional config specified via the command line.
Config file config/plotting.default.yaml is extended by additional config specified via the command line.
Config file config/benchmarking.default.yaml is extended by additional config specified via the command line.
host: mm-mbp-2025
Building DAG of jobs...
Job stats:
job                                           count
------------------------------------------  -------
cba                                               1
clean_projects           

Notice how the `count` of steps changed when we specified a single run, project, and horizon. 
Comparatively, the default settings run the full list of scenarios we have, 5 projects (t4, t16, t28, t33, t35), and two planning horizons (2030 and 2040). So, the full default workflow will run many more steps.

#### Checkpoints

You may notice above that the number of rules is actually quite low.

There is a `checkpoint` in the workflow, called `clean_projects`, that first checks how many projects are being asked to run before building out the full DAG. 
It tells the workflow which CBA projects exist, which project IDs to run, and which method applies (TOOT/PINT). 

Before the `clean_projects` step runs, Snakemake does not know the full list of project jobs it needs to create. 
The step downloads and cleans the external CBA project database, tells the workflow the full list of projects available, which projects the user wants to evaluate, and how each should be evaluated.
After `clean_projects` finishes, Snakemake can read the cleaned CSV and expand the DAG into concrete jobs.

Thus, what we should do first here is run the workflow just up until the checkpoint, then only after that can we run the full workflow again -- in which case, the DAG would show the actual number of jobs that would be run.

In [ ]:
! pixi run tyndp-checkpoint --config 'run={"name":"NT"}' 'cba={"planning_horizons":[2030],"projects":["t4"]}'

⠁ activating environment                                                            snakemake -call cba --configfile config/config.tyndp.yaml --until clean_projects --config run={"name":"NT"} cba={"planning_horizons":[2030],"projects":["t4"]}                               ⠁                                                                               Using workflow specific profile profiles/default for setting default command line arguments.
Config file config/config.default.yaml is extended by additional config specified via the command line.
Config file config/plotting.default.yaml is extended by additional config specified via the command line.
Config file config/benchmarking.default.yaml is extended by additional config specified via the command line.
Assuming unrestricted shared filesystem usage.
Falling back to greedy scheduler because no ILP solver is found (you have to install pulp and either coincbc or glpk).
host: mm-mbp-2025
Building DAG of jobs...
Using shell: /bin/bash
Pro

Now that the checkpoint is complete, we can re-check the DAG of how many steps are needed to run the CBA workflow.

In [ ]:
! pixi run tyndp-cba --config 'run={"name":"NT"}' 'cba={"planning_horizons":[2030],"projects":["t4"]}' -n

⠁ activating environment                                                        kemake -call cba --configfile config/config.tyndp.yaml --config run={"name":"NT"} cba={"planning_horizons":[2030],"projects":["t4"]} -n                                                          ⠁                                                                               Using workflow specific profile profiles/default for setting default command line arguments.
Config file config/config.default.yaml is extended by additional config specified via the command line.
Config file config/plotting.default.yaml is extended by additional config specified via the command line.
Config file config/benchmarking.default.yaml is extended by additional config specified via the command line.
host: mm-mbp-2025
Building DAG of jobs...
Updating checkpoint dependencies.
Job stats:
job                                           count
------------------------------------------  -------
add_electricity                            

### Task X: Change the configuration directly in the `config/config.tyndp.yaml`

- Change the config settings to match above
  - Change scenario name in config.tyndp.yaml
  - Change parameters within NT in scenarios.tyndp.yaml
- Come back here
- Run simplified command line

In [ ]:
# Should give the same output as above, without the arguments
! pixi run tyndp-cba

### Running different and multiple climate years

(Add some info here about how different climate scenarios are defined? Navigate into `config/scenarios.yaml`?)

TODO:
- add snippets of yaml here to show as examples
- can also show live
- stress that in practice it's easier to modify in yaml

First, we again need to run up to the checkpoint for the different climate years:

In [ ]:
! pixi run tyndp-checkpoint --config 'run={"name":["NT-cy2008", "NT-cy2009", "NT-cy1995"]}' 'cba={"planning_horizons":[2030],"projects":["t4"]}'

⠁ activating environment                                                            snakemake -call cba --configfile config/config.tyndp.yaml --until clean_projects --config run={"name":["NT-cy2008", "NT-cy2009", "NT-cy1995"]} cba={"planning_horizons":[2030],"projects":["t4"]}                                                                            ⠁                                                                               Using workflow specific profile profiles/default for setting default command line arguments.
Config file config/config.default.yaml is extended by additional config specified via the command line.
Config file config/plotting.default.yaml is extended by additional config specified via the command line.
Config file config/benchmarking.default.yaml is extended by additional config specified via the command line.
Assuming unrestricted shared filesystem usage.
Falling back to greedy scheduler because no ILP solver is found (you have to install pulp and either coincb

So, we can run for just another climate year, such as the 1995 climate year for the NT scenario (`NT-cy2008`):

In [ ]:
! pixi run tyndp-cba --config 'run={"name":["NT-cy2008"]}' 'cba={"planning_horizons":[2030],"projects":["t4"]}' -n

⠁ activating environment                                                        kemake -call cba --configfile config/config.tyndp.yaml --config run={"name":["NT-cy2008"]} cba={"planning_horizons":[2030],"projects":["t4"]} -n                                                 ⠁                                                                               Using workflow specific profile profiles/default for setting default command line arguments.
Config file config/config.default.yaml is extended by additional config specified via the command line.
Config file config/plotting.default.yaml is extended by additional config specified via the command line.
Config file config/benchmarking.default.yaml is extended by additional config specified via the command line.
host: mm-mbp-2025
Building DAG of jobs...
Updating checkpoint dependencies.
Job stats:
job                                           count
------------------------------------------  -------
add_electricity                            

Or, we can run all climate years (1995, 2008, and 2009) for the `NT` scenario:

In [ ]:
! pixi run tyndp-cba --config 'run={"name":["NT-cyears"]}' 'cba={"planning_horizons":[2030],"projects":["t4"]}' -n

⠁ activating environment                                                        kemake -call cba --configfile config/config.tyndp.yaml --config run={"name":["NT-cyears"]} cba={"planning_horizons":[2030],"projects":["t4"]} -n                                                 ⠁                                                                               Using workflow specific profile profiles/default for setting default command line arguments.
Config file config/config.default.yaml is extended by additional config specified via the command line.
Config file config/plotting.default.yaml is extended by additional config specified via the command line.
Config file config/benchmarking.default.yaml is extended by additional config specified via the command line.
host: mm-mbp-2025
Building DAG of jobs...
Updating checkpoint dependencies.
Job stats:
job                                                    count
---------------------------------------------------  -------
add_electricity          

In [ ]:
! pixi run tyndp-checkpoint --config 'run={"name":["frog"]}' 'cba={"planning_horizons":[2030],"projects":["t4"]}'

⠁ activating environment                                                            snakemake -call cba --configfile config/config.tyndp.yaml --until clean_projects --config run={"name":["frog"]} cba={"planning_horizons":[2030],"projects":["t4"]}                           ⠁                                                                               Using workflow specific profile profiles/default for setting default command line arguments.
Config file config/config.default.yaml is extended by additional config specified via the command line.
Config file config/plotting.default.yaml is extended by additional config specified via the command line.
Config file config/benchmarking.default.yaml is extended by additional config specified via the command line.
Assuming unrestricted shared filesystem usage.
Falling back to greedy scheduler because no ILP solver is found (you have to install pulp and either coincbc or glpk).
host: mm-mbp-2025
Building DAG of jobs...
Using shell: /bin/bash
Pro

In [ ]:
! pixi run tyndp-cba --config 'run={"name":["frog"]}' 'cba={"planning_horizons":[2030],"projects":["t4"]}' -n

⠁ activating environment                                                        kemake -call cba --configfile config/config.tyndp.yaml --config run={"name":["frog"]} cba={"planning_horizons":[2030],"projects":["t4"]} -n                                                      ⠁                                                                               Using workflow specific profile profiles/default for setting default command line arguments.
Config file config/config.default.yaml is extended by additional config specified via the command line.
Config file config/plotting.default.yaml is extended by additional config specified via the command line.
Config file config/benchmarking.default.yaml is extended by additional config specified via the command line.
host: mm-mbp-2025
Building DAG of jobs...
Updating checkpoint dependencies.
Job stats:
job                                           count
------------------------------------------  -------
add_electricity                            

Note how the total count of steps is much higher (especially for CBA-specific steps such as `make_indicators` and weather-dependent steps such as `build_renewable_profiles_pecd`) when multiple climate years are run.

### Task X: Create your own climate year and run it

Go into `scenarios.tyndp.yaml` to create your own climate year run.

Hints:
1. Copy an existing climate year run (e.g., `NT-cy2008`) and modify the details to your liking. Change the name.
2. You would need to first run the new climate year up until the checkpoint.
3. After the checkpoint, you can use `pixi run tyndp-cba` to run the full workflow for your new climate year. 

Warning:
We used previous set of PECD data. So we can only run 1995, 2008, and 2009 at the moment.

Mention the benefit: if we have more climate years in the year, this is how we would add it.

### Running a decoupled SB->CBA Workflow

Based on the DAGs shown in our dry runs, you can see that there are quite a lot of steps and rules that need to be run. A lot of these rules are specific to the SB stage of Open-TYNDP. 

We have the flexibility to bypass running and solving the SB workflow each time, by instead downloading a pre-solved SB network that we have released. By doing this, when running the CBA workflow, we start the process from an already solved SB network, reducing the number of steps significantly.

This can be easily done by setting the `cba.use_presolved` setting to `true`. By default, this downloads and uses the latest release we have (at the time of this workshop, that is v0.7.1).

In [ ]:
! pixi run tyndp-cba --config 'run={"name":["frog"]}' 'cba={"cba_scenario_input":{"use_presolved":true,"sb_version":"latest"},"planning_horizons":[2030],"projects":["t4"]}' -n

⠁ activating environment                                                        kemake -call cba --configfile config/config.tyndp.yaml --config run={"name":["frog"]} cba={"cba_scenario_input":{"use_presolved":true,"sb_version":"latest"},"planning_horizons":[2030],"projects":["t4"]} -n                                                                    ⠁                                                                               Using workflow specific profile profiles/default for setting default command line arguments.
Config file config/config.default.yaml is extended by additional config specified via the command line.
Config file config/plotting.default.yaml is extended by additional config specified via the command line.
Config file config/benchmarking.default.yaml is extended by additional config specified via the command line.
host: mm-mbp-2025
Building DAG of jobs...
Updating checkpoint dependencies.
Job stats:
job                                       count
-------------------

Note how the number of steps has decreased down from over 100 to just ~37, and we see some new steps not previously seen before, such as `retrieve_presolved_sb_networks`. 

### Advanced task: Running and solving a CBA project with a pre-solved network

- Checkout to workshop branch
- Mention warning that it takes 15 minutes to solve
- Let people run it during the break

# extras

In [ ]:
def unzip_with_timestamps(zip_path, extract_to, keep_zip=True):
    """Unzip a file while preserving original file timestamps."""
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        for member in zip_ref.infolist():
            # Extract the file
            zip_ref.extract(member, extract_to)

            # Get the extracted file path
            extracted_path = os.path.join(extract_to, member.filename)

            # Get the modification time from the zip file
            date_time = datetime(*member.date_time)
            timestamp = date_time.timestamp()

            # Set both access and modification times
            os.utime(extracted_path, (timestamp, timestamp))
    if not keep_zip:
        os.remove(zip_path)

In [ ]:
urls = {
    "data/results-0.7.1.zip": "https://storage.googleapis.com/open-tyndp-data-store/outcomes/0.7.1/results-0.7.1.zip",
    "scripts/_helpers.py": "https://raw.githubusercontent.com/open-energy-transition/open-tyndp-workshops/792b8474ab5096e5ab8db2822af4fcd9fe659eb6/open-tyndp-workshops/scripts/_helpers.py",
    "data/open-tyndp-20260604-feat-cba-presolved-networks.zip": "https://storage.googleapis.com/open-tyndp-data-store/workshop-04/open-tyndp-20260604-feat-cba-presolved-networks.zip",
}

os.makedirs("data", exist_ok=True)
os.makedirs("scripts", exist_ok=True)
for name, url in urls.items():
    if os.path.exists(name):
        print(f"File {name} already exists. Skipping download.")
    else:
        print(f"Retrieving {name} from storage.")
        urlretrieve(url, name)
        print(f"File available in {name}.")

to_dir = "data/results-0.7.1"
if not os.path.exists(to_dir):
    print(f"Unzipping data/results-0.7.1.zip.")
    unzip_with_timestamps("data/results-0.7.1.zip", "data/results-0.7.1")
print(f"Latest NT results for Open-TYNDP v0.7.1 are available in '{to_dir}'.")

to_dir = "data/open-tyndp-20260604-feat-cba-presolved-networks"
if not os.path.exists(to_dir):
    print(f"Unzipping data/open-tyndp-20260604-feat-cba-presolved-networks.zip.")
    unzip_with_timestamps(
        "data/open-tyndp-20260604-feat-cba-presolved-networks.zip",
        "data/open-tyndp-20260604-feat-cba-presolved-networks",
    )
print(f"Open-TYNDP available in '{to_dir}'.")


print("Done")

## Using Snakemake to launch the Open-TYNDP workflow

We now need to change our working directory to the Open-TYNDP directory we previously retrieved.

In [ ]:
os.chdir("data/open-tyndp-20260604-feat-cba-presolved-networks")

Be aware that to run the previous section of this notebook, you will need to restore the default working directory using `os.chdir("../../")`.

Let's check that we are indeed in the new directory now:

In [ ]:
os.getcwd()

We can now use Snakemake to call some of the rules to produce outputs with the `open-tyndp` PyPSA model. 

We will use the prepared TYNDP configuration file (`config/config.tyndp.yaml`) and schedule a dry-run with `-n` as we only want to investigate the DAG of the workflow:

In [ ]:
! pixi shell -e open-tyndp

In [ ]:
os.environ["PATH"] = os.path.expanduser("~/.pixi/bin") + ":" + os.environ["PATH"]

In [ ]:
! pixi shell -e open-tyndp

In [ ]:
! snakemake -call cba --configfile config/config.tyndp.yaml -n

As you can see, there is nothing to be done since all necessary outputs are already present in the files. However, we can still explore the set of rules defined in the `Snakefile` and the other `.smk` files. First, we can plot the rule graph, then the full DAG.

The corresponding rule graph to this workflow will look like this:

In [ ]:
! snakemake -call cba --configfile config/config.tyndp.yaml --rulegraph | python ../../scripts/filter_dag.py | dot -Tpng -o rulegraph_open_tyndp.png
# For Windows run instead:
# ! snakemake -call --configfile config/config.tyndp.yaml --rulegraph | python ..\\..\\scripts\\filter_dag.py | dot -Tpng -o rulegraph_open_tyndp.png

In [ ]:
Image("rulegraph_open_tyndp.png")

The corresponding DAG to this workflow will look like this:

In [ ]:
! snakemake -call --configfile config/config.tyndp.yaml --dag | python ../../scripts/filter_dag.py | dot -Tpng -o dag_open_tyndp.png
# For Windows run instead:
# ! snakemake -call --configfile config/config.tyndp.yaml --dag | python ..\\..\\scripts\\filter_dag.py | dot -Tpng -o dag_open_tyndp.png

In [ ]:
Image("dag_open_tyndp.png")

### Apply Manual Adjustments

Open `data/open-tyndp-20251129/config/scenarios.tyndp.yaml` and modify the existing `NT-52SEG-20251129` scenario by adding the following manual adjustments:

- Increase the `marginal_cost` of all H2 imports by a factor of 1.5 (2030) and 1.3 (2040). The supply of imported H2 is included as a `Generator` component with the carrier name `import H2`.
- Change the `efficiency` of `H2 Electrolysis` to 78% for both 2030 and 2040. H2 Electrolysis is added as a `Link` component.
- Remove the initial capacity (`p_nom` and `p_nom_min`) of all `solar-pv-utility` generators for 2030.

Then rerun the model and explore the results in PyPSA-Explorer.

**Hint**: If you need a reminder on running the Snakemake workflow, refer to the section above.

**Hint II**: Always start with a dry run first (add `-n` to your Snakemake command).

**Hint III**: As we only want to solve the model without post-processing, we can call the rule `solve_sector_networks` instead of the `all` rule.

**Hint IV**: Make sure that the scenario that you want to run is specifying a solver that you can use. You can change the solver to Highs (an open-source solver) by changing the following configuration: `solving:solver`.

To run the workflow, we first need to install and activate the `open-tyndp` environment.

The recommended approach for PyPSA models is to use the `pixi` environment manager, which handles all dependencies automatically.

:::{note} 
We are currently working on a robust setup for all operating systems and will focus on this topic specifically in our next workshop.

If you prefer to use conda to install the `open-tyndp` environment locally, refer to our [legacy installation documentation](https://open-tyndp.readthedocs.io/en/latest/installation.html#legacy-method-conda).
:::

To install `pixi` locally on your operating system, follow the steps in the official [pixi installation documentation](https://pixi.sh/dev/installation/).

For execution on Google Colab, install the Linux version of `pixi` in the runtime:

```bash
!wget -qO- https://pixi.sh/install.sh | sh
```

In the Google Colab terminal, you might need to execute:
```bash
exec bash -l
```
for the changes to take effect if pixi is not recognized.

In [ ]:
# Uncomment the next line for running this notebook on Colab
# !wget -qO- https://pixi.sh/install.sh | sh

Next, open a terminal window. Navigate to the `open-tyndp-20251129` repository before launching the workflow:

```bash
cd data/open-tyndp-20251129
```

Once you're in the `open-tyndp-20251129` repository, activate the environment with pixi. This drops you into a shell where you can run the workflow:

```bash
pixi shell -e open-tyndp
```

## Task 1: Run coupled and decoupled SB-CBA workflow

In [ ]:
! pixi run snakemake -call cba --configfile config/config.tyndp.yaml --config 'run={"name":"NT"}' 'data_config="tyndp"' 'cba={"projects":["t4"]}' --until clean_projects -n

In [ ]:
! pixi run snakemake -call cba --configfile config/config.tyndp.yaml --config 'run={"name":"NT"}' 'data_config="tyndp"' 'cba={"projects":["t4"]}' --until clean_projects

In [ ]:
! pixi run tyndp-cba --config 'run={"name":"NT"}' 'data_config="tyndp"' 'cba={"projects":["t4"]}' -n

In [ ]:
! pixi run tyndp-cba --config 'run={"name":"NT-cyears"}' 'data_config="tyndp"' 'cba={"projects":["t4"]}' --until clean_projects

In [ ]:
os.getcwd()

In [ ]:
! pixi run tyndp-cba --config 'run={"name":["NT-cy2008", "NT-cy2009", "NT-cy1995", "NT-cyears"]}' 'data_config="tyndp"' 'cba={"projects":["t4"]}' --until clean_projects

In [ ]:
! pixi run tyndp-cba --config 'run={"name": "NT-cyears"}' 'data_config="tyndp"' 'cba={"projects":["t4"]}' -n

In [ ]:
! pixi run tyndp-cba --config 'run={"name":"NT-cyears"}' 'data_config="tyndp"' 'cba={"projects":["t4"]}' -n

In [ ]:
! pixi run tyndp-cba --config 'run={"name":"NT"}' 'cba={"cba_scenario_input":{"use_presolved":true},"projects":["t4"],"msv_extraction":{"resolution":"48H"},"solving":{"horizon":4381,"overlap":1}}' -n

In [ ]:
! pixi run tyndp-cba --config 'run={"name":"NT"}' 'cba={"cba_scenario_input":{"use_presolved":true},"projects":["t4"],"msv_extraction":{"resolution":"4380H"},"solving":{"horizon":4381,"overlap":1}}'

In [ ]:
! pixi run tyndp-cba --config 'run={"name":"NT"}' 'data_config="tyndp"' 'cba={"projects":["t4"]}'

In [ ]:
! pixi run tyndp-cba --config 'cba={"cba_scenario_input":{"use_presolved":true}}' -n

## Visualize DAG

In [ ]:
os.chdir("/Users/meas/oet/open-tyndp-workshops/open-tyndp-workshops/")

In [ ]:
! snakemake -call --dag | python scripts/filter_dag.py | dot -Tpng -o dag_minimal_2.png